# Spaceship Titanic - Ensemble Pipeline

AI3023 Machine Learning Workshop project. Trains five gradient-boosting and tree-based base models, then compares three ensemble strategies: weighted blend (soft voting), stacking (LR meta-learner), and hard voting (3-way majority).

Best result: hard voting across three diverse single-CatBoost pipelines, public LB 0.81529.

Run cells top to bottom. Set `RUN_TUNING = False` to skip the random search and reuse cached params.

## 1. Imports

In [ ]:
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Libraries loaded.')

## 2. Global Settings

Feature lists and pipeline flags. Set `RUN_TUNING = False` to skip the random search.

In [ ]:
SPEND_COLS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
TARGET_COL = 'Transported'

CAT_FEATURES = [
    'HomePlanet', 'CryoSleep', 'Destination', 'VIP',
    'Deck', 'Side',
    'Is_Solo', 'AgeBin', 'CabinNumBin',
    'HomePlanet_Destination', 'Deck_Side',
]
DROP_COLS = ['PassengerId', 'Cabin', 'Name', 'Group', 'Surname']

# Flags
RUN_TUNING         = True   # set False to skip tuning and load best_tuning_results.json
RUN_FINAL_TRAINING = True

# Tuning budget
TUNING_N_ITER   = 10
TUNING_N_SPLITS = 3

FINAL_N_SPLITS = 5
BLEND_N_TRIALS = 1000

# Voter retrain flag
RUN_VOTERS_FROM_SCRATCH = False

print(f'RUN_TUNING={RUN_TUNING}, RUN_FINAL_TRAINING={RUN_FINAL_TRAINING}')
print(f'RUN_VOTERS_FROM_SCRATCH={RUN_VOTERS_FROM_SCRATCH}')
print(f'Categorical features: {CAT_FEATURES}')

## 3. Exploratory Data Analysis

Target balance, missing values, distributions, categorical effects, the CryoSleep-spending rule, and correlations. The findings drive the feature engineering in Section 4.

In [ ]:
df_train_raw = pd.read_csv('train.csv')
df_test_raw  = pd.read_csv('test.csv')
print('Train shape :', df_train_raw.shape)
print('Test shape  :', df_test_raw.shape)
df_train_raw.head()

In [ ]:
# Target + missing
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

df_train_raw['Transported'].value_counts().plot(
    kind='bar', ax=ax[0], color=['#5b9bd5', '#ed7d31'], rot=0
)
ax[0].set_title('Target Distribution (Transported)')
ax[0].set_ylabel('Count')

missing = df_train_raw.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing.plot(kind='barh', ax=ax[1], color='#7030a0')
ax[1].set_title('Missing values per column (train)')
ax[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Transported True : {int(df_train_raw.Transported.sum())} '
      f'({df_train_raw.Transported.mean()*100:.1f}%)')
print(f'Transported False: {int((~df_train_raw.Transported).sum())}')

In [ ]:
# Numeric distributions
num_cols = ['Age'] + SPEND_COLS
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, num_cols):
    df_train_raw[col].dropna().hist(bins=40, ax=ax, color='#5b9bd5', edgecolor='white')
    ax.set_title(col)
    if col != 'Age':
        ax.set_yscale('log')
plt.suptitle('Numerical Feature Distributions (spending uses log-y)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical rates
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
global_rate = df_train_raw['Transported'].mean()
for ax, col in zip(axes, cat_cols):
    rates = (df_train_raw.groupby(col)['Transported']
                          .mean().sort_values())
    rates.plot(kind='bar', ax=ax, color='#70ad47', rot=0)
    ax.axhline(global_rate, color='red', ls='--', lw=1, label=f'Global = {global_rate:.2f}')
    ax.set_ylim(0, 1)
    ax.set_title(f'Transport rate by {col}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# CryoSleep rule
total = df_train_raw[SPEND_COLS].fillna(0).sum(axis=1)
no_spend  = df_train_raw[total == 0]
yes_spend = df_train_raw[total > 0]

print(f'Passengers with $0 total spending : {len(no_spend)}')
print(f'  CryoSleep=True among these     : {int(no_spend.CryoSleep.fillna(False).sum())}  '
      f'({no_spend.CryoSleep.fillna(False).mean()*100:.1f}%)')
print(f'  Transport rate (no spending)    : {no_spend.Transported.mean()*100:.1f}%')
print()
print(f'Passengers with >$0 spending      : {len(yes_spend)}')
print(f'  CryoSleep=True among these     : {int(yes_spend.CryoSleep.fillna(False).sum())}  '
      f'(should be ~0% — CryoSleep cannot spend)')
print(f'  Transport rate (spending)       : {yes_spend.Transported.mean()*100:.1f}%')

In [ ]:
# Correlation heatmap
corr_data = df_train_raw[['Age'] + SPEND_COLS].copy()
corr_data['Transported'] = df_train_raw['Transported'].astype(int)
corr = corr_data.corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='black' if abs(corr.values[i,j]) < 0.3 else 'white')
plt.colorbar(im, ax=ax)
plt.title('Pearson correlation — numerical features vs Transported')
plt.tight_layout()
plt.show()

### Key findings

1. Balanced target (~50.4% `Transported=True`). Accuracy is a fair metric.
2. Missing rate is ~2% per column. Group-mode and median imputation are enough.
3. Spending columns are right-skewed. We use `log1p` plus aggregates.
4. CryoSleep is the strongest single feature: ~82% transport rate vs ~33%, and `CryoSleep=True` implies zero spending.
5. Europans are transported most, Earthlings least.
6. VIP is weak (only ~2% of passengers).
7. Linear correlations with the target are weak (|rho| < 0.25). Interactions dominate, which favours trees.

## 4. Feature Engineering

Engineered features:

- `Deck`, `Num`, `Side` from `Cabin`
- `Group`, `Group_Size`, `Is_Solo` from the `PassengerId` prefix
- `Total_Spend`, `No_Spend`, `Luxury_Spend`, `Basic_Spend`, `Log_Total_Spend`, `Spend_Ratio`, `Spend_Per_Age`
- `AgeBin` (5 buckets), `CabinNumBin` (5 quantile buckets)
- `Surname`, `Family_Size` from `Name`
- Crosses: `HomePlanet_Destination`, `Deck_Side`

Imputation is logical first (CryoSleep implies zero spending, group-mode for categoricals), then group / median fallback.

In [ ]:
def split_cabin(value):
    """Split Cabin like 'B/0/P' into (deck, num, side)."""
    if pd.isna(value):
        return np.nan, np.nan, np.nan
    parts = str(value).split('/')
    if len(parts) != 3:
        return np.nan, np.nan, np.nan
    deck, num, side = parts
    try:
        num = int(num)
    except ValueError:
        num = np.nan
    return deck, num, side


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    cab = df['Cabin'].apply(split_cabin)
    df['Deck'] = cab.apply(lambda x: x[0])
    df['Num']  = cab.apply(lambda x: x[1])
    df['Side'] = cab.apply(lambda x: x[2])

    df['Group']      = df['PassengerId'].astype(str).str.split('_').str[0]
    df['Group_Size'] = df['Group'].map(df['Group'].value_counts())
    df['Is_Solo']    = (df['Group_Size'] == 1).astype(int)

    df['Total_Spend']    = df[SPEND_COLS].fillna(0).sum(axis=1)
    df['No_Spend']       = (df['Total_Spend'] == 0).astype(int)
    df['Luxury_Spend']   = df[['Spa', 'VRDeck']].fillna(0).sum(axis=1)
    df['Basic_Spend']    = df[['RoomService', 'FoodCourt', 'ShoppingMall']].fillna(0).sum(axis=1)
    df['Log_Total_Spend'] = np.log1p(df['Total_Spend'])
    df['Spend_Ratio']    = df['Luxury_Spend'] / (df['Basic_Spend'] + 1)
    df['Spend_Per_Age']  = df['Total_Spend'] / (df['Age'] + 1)

    df['AgeBin'] = pd.cut(
        df['Age'],
        bins=[-1, 12, 18, 35, 60, 100],
        labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'],
    )
    df['CabinNumBin'] = pd.qcut(df['Num'], q=5, duplicates='drop').astype(str)

    name_parts = df['Name'].fillna('Unknown Unknown').astype(str).str.split(' ')
    df['Surname'] = name_parts.apply(
        lambda x: x[-1] if isinstance(x, list) and len(x) > 1 and x[-1] != 'Unknown' else np.nan
    )
    df['Family_Size'] = df['Surname'].map(df['Surname'].value_counts())

    df['HomePlanet_Destination'] = df['HomePlanet'].astype(str) + '_' + df['Destination'].astype(str)
    df['Deck_Side']              = df['Deck'].astype(str) + '_' + df['Side'].astype(str)

    return df

In [ ]:
def fill_with_mode(series: pd.Series) -> pd.Series:
    mode = series.mode(dropna=True)
    if mode.empty:
        return series
    return series.where(series.notna(), mode.iloc[0])


def logical_impute(df: pd.DataFrame) -> pd.DataFrame:
    """Domain-rule imputation followed by group/median fallbacks."""
    df = df.copy()

 # CryoSleep -> no spend
    for col in SPEND_COLS:
        df.loc[(df['CryoSleep'] == True) & (df[col].isna()), col] = 0

    df['Total_Spend'] = df[SPEND_COLS].fillna(0).sum(axis=1)
    df['No_Spend']    = (df['Total_Spend'] == 0).astype(int)

 # Infer CryoSleep
    df.loc[(df['Total_Spend'] > 0) & (df['CryoSleep'].isna()), 'CryoSleep'] = False
    df.loc[(df['No_Spend']  == 1) & (df['CryoSleep'].isna()), 'CryoSleep'] = True

 # Group-mode fill
    for col in ['HomePlanet', 'Destination', 'VIP']:
        grp_mode = df.groupby('Group')[col].transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan
        )
        df[col] = df[col].fillna(grp_mode)

    for col in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Deck', 'Side']:
        df[col] = fill_with_mode(df[col])

    median_cols = (
        SPEND_COLS + ['Age', 'Num', 'Group_Size', 'Family_Size',
                      'Total_Spend', 'Luxury_Spend', 'Basic_Spend',
                      'Log_Total_Spend', 'Spend_Ratio', 'Spend_Per_Age']
    )
    for col in median_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

 # Recompute spend
    df['Total_Spend']     = df[SPEND_COLS].fillna(0).sum(axis=1)
    df['No_Spend']        = (df['Total_Spend'] == 0).astype(int)
    df['Luxury_Spend']    = df[['Spa', 'VRDeck']].fillna(0).sum(axis=1)
    df['Basic_Spend']     = df[['RoomService', 'FoodCourt', 'ShoppingMall']].fillna(0).sum(axis=1)
    df['Log_Total_Spend'] = np.log1p(df['Total_Spend'])
    df['Spend_Ratio']     = df['Luxury_Spend'] / (df['Basic_Spend'] + 1)
    df['Spend_Per_Age']   = df['Total_Spend'] / (df['Age'] + 1)

    df['AgeBin'] = pd.cut(
        df['Age'],
        bins=[-1, 12, 18, 35, 60, 100],
        labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'],
    )
    df['CabinNumBin']            = pd.qcut(df['Num'].rank(method='first'), q=5, duplicates='drop').astype(str)
    df['HomePlanet_Destination'] = df['HomePlanet'].astype(str) + '_' + df['Destination'].astype(str)
    df['Deck_Side']              = df['Deck'].astype(str) + '_' + df['Side'].astype(str)

    return df


def prepare_datasets(train_path='train.csv', test_path='test.csv'):
    train = pd.read_csv(train_path)
    test  = pd.read_csv(test_path)

    test[TARGET_COL] = np.nan
    combined = pd.concat([train, test], ignore_index=True)

    combined = build_features(combined)
    combined = logical_impute(combined)
    combined = combined.drop(columns=DROP_COLS)

    train_clean = combined[combined[TARGET_COL].notna()].copy()
    test_clean  = combined[combined[TARGET_COL].isna()].copy()

    y = train_clean[TARGET_COL].astype(int)
    X = train_clean.drop(columns=[TARGET_COL])
    X_test = test_clean.drop(columns=[TARGET_COL])

    for col in CAT_FEATURES:
        X[col]      = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    return X, y, X_test, test

## 5. Encoding and Helpers

- `build_encoded_matrices`: one-hot encode categoricals for the non-CatBoost models.
- `to_dense_float32`: HistGradientBoosting needs dense numeric input.
- `search_best_threshold`: grid-search the optimal decision threshold on OOF probabilities.

In [ ]:
def build_encoded_matrices(X_train_fold, X_val_fold, X_test, categorical_cols):
    categorical_cols = [c for c in categorical_cols if c in X_train_fold.columns]
    numeric_cols     = [c for c in X_train_fold.columns if c not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot',  OneHotEncoder(handle_unknown='ignore')),
            ]), categorical_cols),
            ('numeric', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
            ]), numeric_cols),
        ]
    )

    X_train_encoded = preprocessor.fit_transform(X_train_fold)
    X_val_encoded   = preprocessor.transform(X_val_fold)
    X_test_encoded  = preprocessor.transform(X_test)
    return X_train_encoded, X_val_encoded, X_test_encoded


def to_dense_float32(matrix):
    if sparse.issparse(matrix):
        return matrix.toarray().astype(np.float32)
    return np.asarray(matrix, dtype=np.float32)


def search_best_threshold(y_true, prob):
    """Find the OOF-optimal threshold for binary accuracy."""
    best_threshold, best_acc = 0.5, 0.0
    for threshold in np.linspace(0.35, 0.65, 1201):
        pred = (prob >= threshold).astype(int)
        acc  = accuracy_score(y_true, pred)
        if acc > best_acc:
            best_acc, best_threshold = float(acc), float(threshold)
    return best_threshold, best_acc

## 6. Hyperparameter Tuning

Each model has its own `PARAM_SPACE`. `tune_one_model` runs random search with 3-fold CV and reports the best OOF accuracy.

In [ ]:
PARAM_SPACES = {
    'CatBoost': {
        'iterations':         [1200, 1500, 2000, 2500],
        'learning_rate':      [0.02, 0.025, 0.03, 0.035, 0.04],
        'depth':              [5, 6, 7],
        'l2_leaf_reg':        [3.0, 4.0, 5.0, 7.0],
        'random_strength':    [0.5, 1.0, 2.0],
        'bagging_temperature':[0.3, 0.6, 1.0],
    },
    'LightGBM': {
        'n_estimators':       [1200, 1500, 2000, 2500],
        'learning_rate':      [0.02, 0.025, 0.03, 0.035, 0.04],
        'num_leaves':         [15, 31, 47, 63],
        'min_child_samples':  [10, 15, 20, 30, 40],
        'subsample':          [0.75, 0.85, 0.95, 1.0],
        'colsample_bytree':   [0.75, 0.85, 0.95, 1.0],
        'reg_alpha':          [0.0, 0.05, 0.1, 0.3],
        'reg_lambda':         [0.5, 1.0, 1.5, 2.0],
    },
    'XGBoost': {
        'n_estimators':       [1000, 1500, 2000],
        'learning_rate':      [0.02, 0.03, 0.035, 0.04],
        'max_depth':          [3, 4, 5, 6],
        'min_child_weight':   [1, 2, 3, 5],
        'subsample':          [0.75, 0.85, 0.95],
        'colsample_bytree':   [0.75, 0.85, 0.95],
        'gamma':              [0.0, 0.03, 0.05, 0.1],
        'reg_alpha':          [0.0, 0.03, 0.05, 0.1],
        'reg_lambda':         [0.8, 1.0, 1.2, 1.5],
    },
    'ExtraTrees': {
        'n_estimators':       [500, 800, 1200],
        'max_depth':          [None, 8, 12, 16, 20],
        'min_samples_split':  [2, 4, 6, 8],
        'min_samples_leaf':   [1, 2, 3, 4],
        'max_features':       ['sqrt', 'log2', 0.5, 0.8],
        'bootstrap':          [False, True],
    },
    'HistGradientBoosting': {
        'max_iter':           [300, 500, 800],
        'learning_rate':      [0.02, 0.03, 0.035, 0.05],
        'max_leaf_nodes':     [15, 31, 45, 63],
        'min_samples_leaf':   [10, 20, 30, 40],
        'l2_regularization':  [0.0, 0.03, 0.05, 0.1, 0.2],
    },
}

In [ ]:
def evaluate_model_params(model_name, params, X, y, n_splits=3, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_prob = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        if model_name == 'CatBoost':
            m = CatBoostClassifier(
                loss_function='Logloss', eval_metric='Accuracy',
                cat_features=CAT_FEATURES, bootstrap_type='Bayesian',
                verbose=0, random_seed=random_state, allow_writing_files=False,
                **params)
            m.fit(X_tr, y_tr, eval_set=(X_va, y_va),
                  early_stopping_rounds=100, use_best_model=True)
            val_prob = m.predict_proba(X_va)[:, 1]
        else:
            X_tr_e, X_va_e, _ = build_encoded_matrices(X_tr, X_va, X_va, CAT_FEATURES)
            if model_name == 'LightGBM':
                m = lgb.LGBMClassifier(objective='binary', random_state=random_state,
                                       n_jobs=-1, verbose=-1, **params)
                m.fit(X_tr_e, y_tr, eval_set=[(X_va_e, y_va)],
                      callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
                val_prob = m.predict_proba(X_va_e)[:, 1]
            elif model_name == 'XGBoost':
                m = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                                      tree_method='hist', random_state=random_state,
                                      n_jobs=-1, **params)
                m.fit(X_tr_e, y_tr, eval_set=[(X_va_e, y_va)], verbose=False)
                val_prob = m.predict_proba(X_va_e)[:, 1]
            elif model_name == 'ExtraTrees':
                m = ExtraTreesClassifier(random_state=random_state, n_jobs=-1, **params)
                m.fit(X_tr_e, y_tr)
                val_prob = m.predict_proba(X_va_e)[:, 1]
            elif model_name == 'HistGradientBoosting':
                X_tr_d = to_dense_float32(X_tr_e); X_va_d = to_dense_float32(X_va_e)
                m = HistGradientBoostingClassifier(
                    early_stopping=True, validation_fraction=0.15,
                    n_iter_no_change=50, random_state=random_state, **params)
                m.fit(X_tr_d, y_tr)
                val_prob = m.predict_proba(X_va_d)[:, 1]
            else:
                raise ValueError(f'Unknown model: {model_name}')
        oof_prob[val_idx] = val_prob

    return search_best_threshold(y.values, oof_prob)


def tune_one_model(model_name, X, y, n_iter=10, n_splits=3, random_state=42):
    print(f'\n===== Tuning {model_name} =====')
    sampler = list(ParameterSampler(PARAM_SPACES[model_name], n_iter=n_iter,
                                    random_state=random_state))
    best_score, best_threshold, best_params = -1.0, 0.5, None
    for i, params in enumerate(sampler, start=1):
        threshold, score = evaluate_model_params(model_name, params, X, y,
                                                 n_splits=n_splits,
                                                 random_state=random_state)
        flag = ' ← new best' if score > best_score else ''
        print(f'[{i:02d}/{n_iter}] OOF Acc = {score:.4f}  threshold = {threshold:.4f}{flag}')
        if score > best_score:
            best_score, best_threshold, best_params = score, threshold, params
    print(f'Best {model_name}: {best_score:.4f}  thr={best_threshold:.4f}')
    return best_params, best_score, best_threshold


def tune_all_models(X, y, n_iter=10, n_splits=3):
    results = {}
    for name in ['CatBoost', 'LightGBM', 'XGBoost', 'ExtraTrees', 'HistGradientBoosting']:
        best_params, best_score, best_threshold = tune_one_model(
            name, X, y, n_iter=n_iter, n_splits=n_splits)
        results[name] = {'best_params': best_params,
                         'best_score': best_score,
                         'best_threshold': best_threshold}
    with open('best_tuning_results.json', 'w') as f:
        json.dump(results, f, indent=4)
    return results

## 7. Final 5-Fold Training

Trains all five base models with `StratifiedKFold(n=5)`. Returns OOF probability vectors and averaged test probability vectors per model.

In [ ]:
def train_and_predict(X, y, X_test, n_splits=5, random_state=42, tuned_params=None):
    if tuned_params is None:
        tuned_params = {}

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model_names = ['CatBoost', 'LightGBM', 'XGBoost', 'ExtraTrees', 'HistGradientBoosting']

    oof_probs  = {name: np.zeros(len(X))      for name in model_names}
    test_preds = {name: np.zeros(len(X_test)) for name in model_names}
    fold_acc   = {name: [] for name in model_names}

    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        print(f'\n--- FOLD {fold} ---')
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]

        X_tr_e, X_va_e, X_te_e = build_encoded_matrices(X_tr, X_va, X_test, CAT_FEATURES)
        X_tr_d = to_dense_float32(X_tr_e)
        X_va_d = to_dense_float32(X_va_e)
        X_te_d = to_dense_float32(X_te_e)

 # CatBoost
        cb_p = {'iterations': 1500, 'learning_rate': 0.035, 'depth': 6,
                'l2_leaf_reg': 4.0, 'random_strength': 1.0,
                'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.6}
        cb_p.update(tuned_params.get('CatBoost', {}))
        cb = CatBoostClassifier(loss_function='Logloss', eval_metric='Accuracy',
                                cat_features=CAT_FEATURES, verbose=0,
                                random_seed=random_state, allow_writing_files=False, **cb_p)
        cb.fit(X_tr, y_tr, eval_set=(X_va, y_va),
               early_stopping_rounds=100, use_best_model=True)
        p_val = cb.predict_proba(X_va)[:, 1]
        oof_probs['CatBoost'][va] = p_val
        test_preds['CatBoost']   += cb.predict_proba(X_test)[:, 1] / n_splits
        fold_acc['CatBoost'].append(accuracy_score(y_va, (p_val >= 0.5).astype(int)))

 # LightGBM
        lgb_p = {'n_estimators': 1500, 'learning_rate': 0.035, 'num_leaves': 31,
                 'max_depth': -1, 'min_child_samples': 20,
                 'subsample': 0.85, 'colsample_bytree': 0.85,
                 'reg_alpha': 0.1, 'reg_lambda': 1.0}
        lgb_p.update(tuned_params.get('LightGBM', {}))
        lgbm = lgb.LGBMClassifier(objective='binary', random_state=random_state,
                                  n_jobs=-1, verbose=-1, **lgb_p)
        lgbm.fit(X_tr_e, y_tr, eval_set=[(X_va_e, y_va)],
                 callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
        p_val = lgbm.predict_proba(X_va_e)[:, 1]
        oof_probs['LightGBM'][va] = p_val
        test_preds['LightGBM']   += lgbm.predict_proba(X_te_e)[:, 1] / n_splits
        fold_acc['LightGBM'].append(accuracy_score(y_va, (p_val >= 0.5).astype(int)))

 # XGBoost
        xgb_p = {'n_estimators': 1500, 'learning_rate': 0.035, 'max_depth': 5,
                 'min_child_weight': 2, 'subsample': 0.85, 'colsample_bytree': 0.85,
                 'gamma': 0.05, 'reg_alpha': 0.05, 'reg_lambda': 1.0}
        xgb_p.update(tuned_params.get('XGBoost', {}))
        xgbm = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                                 tree_method='hist', random_state=random_state,
                                 n_jobs=-1, **xgb_p)
        xgbm.fit(X_tr_e, y_tr, eval_set=[(X_va_e, y_va)], verbose=False)
        p_val = xgbm.predict_proba(X_va_e)[:, 1]
        oof_probs['XGBoost'][va] = p_val
        test_preds['XGBoost']   += xgbm.predict_proba(X_te_e)[:, 1] / n_splits
        fold_acc['XGBoost'].append(accuracy_score(y_va, (p_val >= 0.5).astype(int)))

 # ExtraTrees
        et_p = {'n_estimators': 800, 'max_depth': None, 'min_samples_split': 4,
                'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': False}
        et_p.update(tuned_params.get('ExtraTrees', {}))
        et = ExtraTreesClassifier(random_state=random_state, n_jobs=-1, **et_p)
        et.fit(X_tr_e, y_tr)
        p_val = et.predict_proba(X_va_e)[:, 1]
        oof_probs['ExtraTrees'][va] = p_val
        test_preds['ExtraTrees']   += et.predict_proba(X_te_e)[:, 1] / n_splits
        fold_acc['ExtraTrees'].append(accuracy_score(y_va, (p_val >= 0.5).astype(int)))

 # HistGradientBoosting
        hgb_p = {'max_iter': 500, 'learning_rate': 0.035, 'max_leaf_nodes': 31,
                 'min_samples_leaf': 20, 'l2_regularization': 0.05}
        hgb_p.update(tuned_params.get('HistGradientBoosting', {}))
        hgb = HistGradientBoostingClassifier(early_stopping=True,
                                             validation_fraction=0.15,
                                             n_iter_no_change=50,
                                             random_state=random_state, **hgb_p)
        hgb.fit(X_tr_d, y_tr)
        p_val = hgb.predict_proba(X_va_d)[:, 1]
        oof_probs['HistGradientBoosting'][va] = p_val
        test_preds['HistGradientBoosting']   += hgb.predict_proba(X_te_d)[:, 1] / n_splits
        fold_acc['HistGradientBoosting'].append(accuracy_score(y_va, (p_val >= 0.5).astype(int)))

        for name in model_names:
            print(f'  {name:>22}: fold acc = {fold_acc[name][-1]:.4f}')

    return oof_probs, test_preds, fold_acc

## 8. Ensemble Strategies

- Single submissions: one CSV per base model using its OOF-optimal threshold.
- Weighted Blend (soft voting): 1000 Dirichlet samples; pick the weights with the highest OOF accuracy.
- Stacking: OOF probabilities as features, with a LogisticRegression meta-learner fit by 5-fold CV.

In [ ]:
def save_individual_submissions(test_preds, oof_probs, y, raw_test):
    for name, probs in test_preds.items():
        thr, _ = search_best_threshold(y.values, oof_probs[name])
        sub = pd.DataFrame({'PassengerId': raw_test['PassengerId'],
                            'Transported': (probs >= thr).astype(bool)})
        sub.to_csv(f'submission_{name.lower()}.csv', index=False)
        print(f'  saved submission_{name.lower()}.csv (thr={thr:.4f})')


def search_best_blend_weights(oof_probs, y, n_trials=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    names = list(oof_probs.keys())
    best_acc, best_w, best_thr = 0.0, None, 0.5
    for _ in range(n_trials):
        w = rng.dirichlet(np.ones(len(names)))
        blend = sum(wi * oof_probs[n] for wi, n in zip(w, names))
        thr, acc = search_best_threshold(y.values, blend)
        if acc > best_acc:
            best_acc, best_w, best_thr = acc, dict(zip(names, w)), thr
    return best_w, best_thr, best_acc


def save_weighted_blend_submission(test_preds, oof_probs, y, raw_test):
    w, thr, acc = search_best_blend_weights(oof_probs, y, n_trials=BLEND_N_TRIALS)
    blend_test = sum(wi * test_preds[n] for n, wi in w.items())
    sub = pd.DataFrame({'PassengerId': raw_test['PassengerId'],
                        'Transported': (blend_test >= thr).astype(bool)})
    sub.to_csv('submission_weighted_blend.csv', index=False)

    print(f'\nWeighted Blend OOF accuracy : {acc:.4f}  (thr={thr:.4f})')
    print('Best weights:')
    for n, wi in w.items():
        print(f'  {n:>22}: {wi:.4f}')
    return w, thr, acc


def save_stacking_submission(test_preds, oof_probs, y, raw_test):
    names = list(oof_probs.keys())
    X_meta      = np.column_stack([oof_probs[n] for n in names])
    X_meta_test = np.column_stack([test_preds[n] for n in names])

    meta_oof  = np.zeros(len(y))
    meta_test = np.zeros(len(raw_test))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for tr, va in skf.split(X_meta, y):
        lr = LogisticRegression(C=1.0, max_iter=1000)
        lr.fit(X_meta[tr], y.iloc[tr])
        meta_oof[va] = lr.predict_proba(X_meta[va])[:, 1]
        meta_test  += lr.predict_proba(X_meta_test)[:, 1] / skf.n_splits

    thr, acc = search_best_threshold(y.values, meta_oof)
    sub = pd.DataFrame({'PassengerId': raw_test['PassengerId'],
                        'Transported': (meta_test >= thr).astype(bool)})
    sub.to_csv('submission_stacking.csv', index=False)

 # Final coefs
    lr_final = LogisticRegression(C=1.0, max_iter=1000)
    lr_final.fit(X_meta, y)
    coefs = dict(zip(names, lr_final.coef_[0]))

    print(f'\nStacking OOF accuracy : {acc:.4f}  (thr={thr:.4f})')
    print('Meta-model coefficients:')
    for n, c in coefs.items():
        print(f'  {n:>22}: {c:+.4f}')
    return thr, acc, coefs

## 9. Run the Pipeline

Loads the data, runs FE and imputation, tunes hyperparameters (if `RUN_TUNING`), trains the five base models with 5-fold CV, and saves all submissions.

In [ ]:
X, y, X_test, raw_test = prepare_datasets()
print('X shape     :', X.shape)
print('X_test shape:', X_test.shape)
X.head()

In [ ]:
tuned_params = {}
if RUN_TUNING:
    print('Running hyperparameter tuning... (~30-50 min on CPU)')
    best_results = tune_all_models(X, y, n_iter=TUNING_N_ITER, n_splits=TUNING_N_SPLITS)
    tuned_params = {k: v['best_params'] for k, v in best_results.items()}
elif os.path.exists('best_tuning_results.json'):
    with open('best_tuning_results.json') as f:
        best_results = json.load(f)
    tuned_params = {k: v['best_params'] for k, v in best_results.items()}
    print('Loaded cached best tuning results from best_tuning_results.json')

for name, p in tuned_params.items():
    print(f'\n{name:>22}: {p}')

In [ ]:
if RUN_FINAL_TRAINING:
    oof_probs, test_preds, fold_acc = train_and_predict(
        X, y, X_test,
        n_splits=FINAL_N_SPLITS,
        random_state=42,
        tuned_params=tuned_params,
    )

In [ ]:
# OOF summary
rows = []
for name, probs in oof_probs.items():
    cv_05  = np.mean(fold_acc[name])
    thr, acc = search_best_threshold(y.values, probs)
    rows.append([name, cv_05, thr, acc])
oof_summary = pd.DataFrame(rows, columns=['Model', 'CV Acc @ 0.5',
                                          'Best Threshold', 'OOF Acc @ Best Thr'])
oof_summary

In [ ]:
print('Saving individual model submissions...')
save_individual_submissions(test_preds, oof_probs, y, raw_test)

In [ ]:
print('Strategy ① — Weighted Blend (Soft Voting)')
blend_w, blend_thr, blend_acc = save_weighted_blend_submission(
    test_preds, oof_probs, y, raw_test
)

In [ ]:
print('Strategy ② — Stacking (LR Meta-Learner)')
stack_thr, stack_acc, stack_coefs = save_stacking_submission(
    test_preds, oof_probs, y, raw_test
)

## 10. Hard Voting - 3-Way Majority

Soft voting and stacking rely on calibrated probabilities. On this dataset, hard voting across three diverse single-CatBoost pipelines beats both. The three voters use different feature-engineering pipelines, which decorrelates their errors and lets majority voting recover accuracy.

This section reproduces each voter end-to-end and combines them.

| Sub-section | Voter | Distinctive FE | Public LB |
| --- | --- | --- | --- |
| 10.1 | v9 - Optuna CatBoost, minimal FE | cabin split, group, spend aggregates, log spend, Is_Child; drops Name, AgeBin, CabinNumBin | 0.81365 |
| 10.2 | p5t6 - RandomSearch CatBoost, rich FE | adds Name_first / Name_last LabelEncoded, random-sample imputation, top-27 feature selection | 0.81458 |
| 10.3 | baseline - Default CatBoost, basic FE | Kaggle reference, single 70/30 split, hand-picked params | 0.81248 |
| 10.4 | Majority Vote | per-row 2-of-3 binary majority | 0.81529 |

`RUN_VOTERS_FROM_SCRATCH = False` skips retraining when the voter CSVs already exist. Set to `True` for a clean reproduction.

### 10.1 Voter 1 - v9 (Optuna CatBoost, minimal FE)

Keeps only the strongest engineered features: cabin parts, group size, total and log spend, the `NoSpending` flag, `LuxurySpend = Spa + VRDeck`, and `Is_Child`. Drops Name entirely (no surname, no AgeBin, no CabinNumBin).

Optuna best params are hardcoded below. Training: 5 seeds x 5-fold StratifiedKFold = 25 fits with probability averaging. Each fit uses 80% of training rows with `early_stopping_rounds=100`.

In [ ]:
# Optuna best params
V9_BEST_PARAMS = dict(
    learning_rate=0.07485688984060579,
    depth=6,
    l2_leaf_reg=1.316777848260815,
    bagging_temperature=0.11773270507332459,
    random_strength=8.488957933498611,
    border_count=205,
)
V9_SEEDS = [42, 7, 2024, 1234, 99]


def v9_split_cabin(value):
    if pd.isna(value):
        return 'U', np.nan, 'U'
    parts = str(value).split('/')
    if len(parts) != 3:
        return 'U', np.nan, 'U'
    deck, num, side = parts
    try:
        num = int(num)
    except ValueError:
        num = np.nan
    return deck, num, side


def v9_feature_engineer(train_df, test_df):
    """Minimal baseline-faithful FE used by voter 1 (mirrors v9.py)."""
    out = []
    for df in [train_df.copy(), test_df.copy()]:
        cab = df['Cabin'].apply(v9_split_cabin)
        df['Deck'] = cab.apply(lambda x: x[0])
        df['Num']  = cab.apply(lambda x: x[1])
        df['Side'] = cab.apply(lambda x: x[2])
        df['Group'] = df['PassengerId'].astype(str).str.split('_').str[0]
        out.append(df)
    df_tr_v, df_te_v = out

    combined = pd.concat([df_tr_v, df_te_v], ignore_index=True)
    for col in ['HomePlanet', 'Destination']:
        grp_mode = combined.groupby('Group')[col].transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
        combined[col] = combined[col].fillna(grp_mode)
        combined[col] = combined[col].fillna(combined[col].mode()[0])
    n_train = len(df_tr_v)
    df_tr_v[['HomePlanet','Destination']] = combined.iloc[:n_train][['HomePlanet','Destination']].values
    df_te_v[['HomePlanet','Destination']]  = combined.iloc[n_train:][['HomePlanet','Destination']].values

    for df in [df_tr_v, df_te_v]:
        df[SPEND_COLS] = df[SPEND_COLS].fillna(0)
        df['TotalSpend']  = df[SPEND_COLS].sum(axis=1)
        df['NoSpending']  = (df['TotalSpend'] == 0).astype(int)
        df['LuxurySpend'] = df['Spa'] + df['VRDeck']

    for df in [df_tr_v, df_te_v]:
        df.loc[df['CryoSleep'].isna() & (df['TotalSpend'] > 0), 'CryoSleep'] = False
    cryo_mode = df_tr_v['CryoSleep'].mode()[0]
    for df in [df_tr_v, df_te_v]:
        df['CryoSleep'] = df['CryoSleep'].fillna(cryo_mode)
        df['Age'] = df['Age'].fillna(df_tr_v['Age'].median())
        df['VIP'] = df['VIP'].fillna(df_tr_v['VIP'].mode()[0])
        num = pd.to_numeric(df['Num'], errors='coerce')
        df['Num'] = num.fillna(num.median())
        df['Group_Size'] = df.groupby('Group')['Group'].transform('count')
        df['Is_Child'] = (df['Age'] <= 12).astype(int)
        for c in SPEND_COLS + ['TotalSpend']:
            df[f'{c}_Log'] = np.log1p(df[c])

    drop_cols = ['PassengerId', 'Cabin', 'Name', 'Group']
    test_id_v = df_te_v['PassengerId'].copy()
    df_tr_v = df_tr_v.drop(columns=drop_cols)
    df_te_v = df_te_v.drop(columns=drop_cols)
    df_tr_v = pd.get_dummies(df_tr_v, columns=['HomePlanet','Destination','Side','Deck'])
    df_te_v = pd.get_dummies(df_te_v,  columns=['HomePlanet','Destination','Side','Deck'])
    df_te_v = df_te_v.reindex(
        columns=[c for c in df_tr_v.columns if c != 'Transported'],
        fill_value=False)
    for col in ['CryoSleep', 'VIP']:
        df_tr_v[col] = df_tr_v[col].astype(int)
        df_te_v[col] = df_te_v[col].astype(int)
    return df_tr_v, df_te_v, test_id_v

In [ ]:
# Train voter 1
from sklearn.preprocessing import StandardScaler

if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('submission_v9.csv'):
    train_raw_v9 = pd.read_csv('train.csv')
    test_raw_v9  = pd.read_csv('test.csv')
    df_tr_v9, df_te_v9, test_id_v9 = v9_feature_engineer(train_raw_v9, test_raw_v9)

    y_v9 = df_tr_v9['Transported'].astype(int).reset_index(drop=True)
    X_v9 = df_tr_v9.drop(columns=['Transported']).reset_index(drop=True)
    X_v9_test = df_te_v9.reset_index(drop=True)

    sc_v9 = StandardScaler()
    X_v9s      = pd.DataFrame(sc_v9.fit_transform(X_v9), columns=X_v9.columns)
    X_v9s_test = pd.DataFrame(sc_v9.transform(X_v9_test), columns=X_v9.columns)

    cb_p = dict(V9_BEST_PARAMS, iterations=3000,
                loss_function='Logloss', eval_metric='Accuracy', verbose=0)

    t0 = time.time()
    test_avg_v9 = np.zeros(len(X_v9_test))
    n_fits = len(V9_SEEDS) * 5
    for si, seed in enumerate(V9_SEEDS):
        skf_v9 = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold, (tr_i, va_i) in enumerate(skf_v9.split(X_v9s, y_v9), start=1):
            cb_v9 = CatBoostClassifier(**dict(cb_p, random_seed=seed),
                                       allow_writing_files=False)
            cb_v9.fit(X_v9s.iloc[tr_i], y_v9.iloc[tr_i],
                      eval_set=(X_v9s.iloc[va_i], y_v9.iloc[va_i]),
                      early_stopping_rounds=100, verbose=0)
            test_avg_v9 += cb_v9.predict_proba(X_v9s_test)[:, 1] / n_fits
        print(f'  seed {seed} done   ({(time.time()-t0)/60:.1f} min elapsed)')

    v9_pred = (test_avg_v9 > 0.5).astype(bool)
    pd.DataFrame({'PassengerId': test_id_v9, 'Transported': v9_pred}
                 ).to_csv('submission_v9.csv', index=False)
    print(f'\nVoter 1 done. Saved submission_v9.csv   ({test_avg_v9.shape[0]} rows)')
else:
    print('submission_v9.csv already exists — skipping voter-1 retraining.')
    print('Set RUN_VOTERS_FROM_SCRATCH = True in Section 2 to force retraining.')

### 10.2 Voter 2 - p5t6 (RandomSearch CatBoost, rich FE)

The richest of the three FE pipelines:

- Keeps `Name` as `Name_first` and `Name_last` LabelEncoded features (implicit family signal).
- Random-sample imputation for `HomePlanet` and `Destination`, weighted by their empirical frequencies. Random F/G or S/P assignment for missing Cabin parts.
- Log transform of the spend columns with zero replaced by 0.367 to avoid `log(0)`.
- Runs a baseline CatBoost feature-importance ranking, then keeps the top-27 features.
- Final fit on the top-27 features with the RandomSearch best params from the original notebook.

In [ ]:
# RandomSearch best params
P5T6_PARAMS = dict(
    learning_rate=0.018049356549743555,
    depth=6,
    l2_leaf_reg=7.50,
    border_count=182,
    random_seed=42,
)


def p5t6_prepare(seed=42):
    """End-to-end FE + imputation reproducing the p5t6 notebook."""
    from sklearn.preprocessing import LabelEncoder
    rng = np.random.RandomState(seed)

    df_tr = pd.read_csv('train.csv')
    df_te = pd.read_csv('test.csv')
    df = pd.concat([df_tr, df_te], axis=0)

    base_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpending'] = df[base_cols].sum(axis=1)

 # Split Cabin
    cab = df['Cabin'].apply(
        lambda x: x.split('/') if not isinstance(x, float) else ['-1','-1','-1']).tolist()
    cab = np.array(cab)
    df['Cabin_deck'] = cab[:, 0]
    df['Cabin_num']  = cab[:, 1]
    df['Cabin_side'] = cab[:, 2]
    df.drop(columns='Cabin', inplace=True)

 # Group size
    grp = df['PassengerId'].apply(lambda x: x.split('_')[0]).value_counts().to_dict()
    df['Group_size'] = df['PassengerId'].apply(lambda x: grp[x.split('_')[0]])
    df.set_index('PassengerId', inplace=True)

 # Split Name
    df['Name'] = df['Name'].fillna('Unk Unk')
    parts = np.array(df['Name'].apply(lambda x: x.split(' ')).tolist())
    df['Name_first'] = parts[:, 0]
    df['Name_last']  = parts[:, 1]
    df.drop(columns=['Name'], inplace=True)

 # Random-sample fill
    for c in ['HomePlanet', 'Destination']:
        vc = df[c].value_counts()
        df.loc[df[c].isna(), c] = rng.choice(
            vc.index, df[c].isna().sum(), p=(vc / vc.sum()).values)

 # Cabin random fill
    n_miss = (df['Cabin_deck'] == '-1').sum()
    df.loc[df['Cabin_deck'] == '-1', 'Cabin_deck'] = rng.choice(
        ['F', 'G'], n_miss, p=[0.5, 0.5])
    df['Cabin_num'] = pd.to_numeric(df['Cabin_num'], errors='coerce')
    mean_num = df['Cabin_num'].mean()
    df['Cabin_num'] = df['Cabin_num'].fillna(int(mean_num)).astype(int)
    n_miss = (df['Cabin_side'] == '-1').sum()
    df.loc[df['Cabin_side'] == '-1', 'Cabin_side'] = rng.choice(
        ['S', 'P'], n_miss, p=[0.5, 0.5])
    df['Cabin_side'] = df['Cabin_side'].map({'S': 0, 'P': 1})

 # Age random fill
    mean_age, std_age = df['Age'].mean(), df['Age'].std()
    nan_mask = df['Age'].isna()
    df.loc[nan_mask, 'Age'] = rng.uniform(
        mean_age - std_age, mean_age + std_age, nan_mask.sum())

 # CryoSleep fill
    df.loc[df['CryoSleep'].isna() & (df['TotalSpending'] != 0), 'CryoSleep'] = False
    df['CryoSleep'] = df['CryoSleep'].fillna(False)

 # Force zero spend
    spend_cols = base_cols + ['TotalSpending']
    for c in spend_cols:
        df.loc[(df['CryoSleep'] == True) & df[c].isna(), c] = 0.0

 # Zero pattern
    for col in base_cols:
        others = [c for c in base_cols if c != col]
        zero_others = (df[others] == 0).all(axis=1)
        df.loc[zero_others & df[col].isna(), col] = 0.0

 # Median fallback
    for c in base_cols:
        df[c] = df[c].fillna(df[c].median())
    df['TotalSpending'] = df[base_cols].sum(axis=1)

 # Log transform
    for c in spend_cols:
        df.loc[df[c] == 0, c] = 0.367
        df[c] = np.log(df[c])

    df['VIP'] = df['VIP'].fillna(False)
    df['CryoSleep'] = df['CryoSleep'].astype(bool)

 # Label-encode names
    df['Name_first'] = LabelEncoder().fit_transform(df['Name_first'])
    df['Name_last']  = LabelEncoder().fit_transform(df['Name_last'])

 # One-hot encode
    df = pd.concat([df, pd.get_dummies(df[['HomePlanet','Destination','Cabin_deck']])], axis=1)
    df.drop(columns=['HomePlanet','Destination','Cabin_deck'], inplace=True)

    df_tr_p = df.iloc[:len(df_tr)].copy()
    df_te_p = df.iloc[len(df_tr):].copy()
    df_tr_p['Transported'] = df_tr_p['Transported'].astype(bool)

    y_p   = df_tr_p['Transported']
    X_p   = df_tr_p.drop(columns=['Transported'])
    X_pte = df_te_p.drop(columns=['Transported'])
    return X_p, y_p, X_pte

In [ ]:
# Train voter 2
if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('final_spaceship_submission.csv'):
    from sklearn.model_selection import train_test_split

    np.random.seed(42)
    X_p, y_p, X_pte = p5t6_prepare(seed=42)

    X_tr_p, X_va_p, y_tr_p, y_va_p = train_test_split(
        X_p, y_p, test_size=0.15, random_state=42)

 # Feature importance
    imp_model = CatBoostClassifier(verbose=False, allow_writing_files=False)
    imp_model.fit(X_tr_p, y_tr_p)
    imp_df = pd.DataFrame({'feature': X_tr_p.columns,
                           'importance': imp_model.feature_importances_})
    final_features = list(
        imp_df.sort_values('importance', ascending=False).head(27)['feature'])
    print(f'Voter-2 top-27 features (by CatBoost importance):')
    for f in final_features:
        print(f'  • {f}')

    final_train = X_tr_p[final_features]
    final_test  = X_pte[final_features]

    cb_p2 = CatBoostClassifier(**P5T6_PARAMS, verbose=False, allow_writing_files=False)
    cb_p2.fit(final_train, y_tr_p)
    y_pred_p = cb_p2.predict(final_test).astype(bool).ravel()

    pd.DataFrame({'PassengerId': X_pte.index, 'Transported': y_pred_p}
                 ).to_csv('final_spaceship_submission.csv', index=False)
    print(f'\nVoter 2 done. Saved final_spaceship_submission.csv   ({len(y_pred_p)} rows)')
else:
    print('final_spaceship_submission.csv already exists — skipping voter-2 retraining.')
    print('Set RUN_VOTERS_FROM_SCRATCH = True in Section 2 to force retraining.')

### 10.3 Voter 3 - baseline (default CatBoost, basic FE)

The Kaggle reference implementation. The FE is close to voter 1's minimal FE, but the training uses a single 70/30 train/val split (`random_state=42`) and the hand-picked CatBoost defaults from the reference notebook: `lr=0.01`, `iter=2000`, `depth=6`, `l2=3`. No CV averaging, no Optuna.

In [ ]:
# Reference baseline
BASELINE_PARAMS = dict(
    iterations=2000,
    learning_rate=0.01,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=False,
)


def baseline_prepare():
    train_raw = pd.read_csv('train.csv')
    test_raw  = pd.read_csv('test.csv')

    test_raw['Transported'] = np.nan
    df = pd.concat([train_raw, test_raw], axis=0)

    df[['Deck','Num','Side']] = df['Cabin'].str.split('/', expand=True)
    df['TotalSpend'] = df[SPEND_COLS].sum(axis=1)
    df['Group']      = df['PassengerId'].apply(lambda x: x.split('_')[0])
    df['Group_Size'] = df.groupby('Group')['Group'].transform('count')

    for c in ['HomePlanet', 'Destination']:
        grp_mode = df.groupby('Group')[c].transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
        df[c] = df[c].fillna(grp_mode)
        df[c] = df[c].fillna(df[c].mode()[0])

    df.loc[df['CryoSleep'].isna() & (df['TotalSpend'] > 0), 'CryoSleep'] = False
    df['CryoSleep'] = df['CryoSleep'].fillna(df['CryoSleep'].mode()[0])

    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])

    df[SPEND_COLS] = df[SPEND_COLS].fillna(0)
    df['TotalSpend']  = df[SPEND_COLS].sum(axis=1)
    df['NoSpending']  = (df['TotalSpend'] == 0).astype(int)
    df['LuxurySpend'] = df['Spa'] + df['VRDeck']
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['VIP'] = df['VIP'].fillna(df['VIP'].mode()[0])
    df['Num'] = pd.to_numeric(df['Num'], errors='coerce')
    df['Num'] = df['Num'].fillna(df['Num'].median())

    df['Group_Size'] = df.groupby('Group')['Group'].transform('count')
    df['Is_Child']   = (df['Age'] <= 12).astype(int)
    for c in SPEND_COLS + ['TotalSpend']:
        df[f'{c}_Log'] = np.log1p(df[c])

    test_id_b = test_raw['PassengerId'].copy()
    df.drop(columns=['Cabin','Name','Group','PassengerId'], inplace=True)
    df = pd.get_dummies(df, columns=['HomePlanet','Destination','Side','Deck'])
    for c in ['CryoSleep','VIP']:
        df[c] = df[c].astype(int)

    df_tr_b = df[df['Transported'].notna()].copy()
    df_te_b = df[df['Transported'].isna()].copy().drop(columns=['Transported'])
    df_tr_b['Transported'] = df_tr_b['Transported'].astype(int)
    return df_tr_b, df_te_b, test_id_b


if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('submission_with_cady_new.csv'):
    from sklearn.model_selection import train_test_split as tts
    from sklearn.preprocessing import StandardScaler as _SS

    df_tr_b, df_te_b, test_id_b = baseline_prepare()
    y_b = df_tr_b['Transported']
    X_b = df_tr_b.drop(columns=['Transported'])

    X_tr_b, X_va_b, y_tr_b, y_va_b = tts(X_b, y_b, test_size=0.3, random_state=42)
    sc_b = _SS()
    X_tr_b_s = sc_b.fit_transform(X_tr_b)
    df_te_b_s = sc_b.transform(df_te_b)

    cb_b = CatBoostClassifier(**BASELINE_PARAMS, random_seed=42, allow_writing_files=False)
    cb_b.fit(X_tr_b_s, y_tr_b)
    y_pred_b = cb_b.predict(df_te_b_s).astype(bool).ravel()

    pd.DataFrame({'PassengerId': test_id_b, 'Transported': y_pred_b}
                 ).to_csv('submission_with_cady_new.csv', index=False)
    print(f'Voter 3 done. Saved submission_with_cady_new.csv   ({len(y_pred_b)} rows)')
else:
    print('submission_with_cady_new.csv already exists — skipping voter-3 retraining.')
    print('Set RUN_VOTERS_FROM_SCRATCH = True in Section 2 to force retraining.')

### 10.4 Combine - 3-Way Majority Vote

For each test row, count how many of {v9, p5t6, baseline} predict `Transported = True`. The majority (at least 2 of 3) is the final label. Public LB 0.81529, saved to `submission_v10_majority.csv`.

In [ ]:
V9_CSV   = 'submission_v9.csv'
SN_CSV   = 'final_spaceship_submission.csv'
BASE_CSV = 'submission_with_cady_new.csv'

v9_sub   = pd.read_csv(V9_CSV).rename(columns={'Transported': 'v9'})
sn_sub   = pd.read_csv(SN_CSV).rename(columns={'Transported': 'sn'})
base_sub = pd.read_csv(BASE_CSV).rename(columns={'Transported': 'base'})

merged = v9_sub.merge(sn_sub, on='PassengerId').merge(base_sub, on='PassengerId')

# Majority vote
votes = merged['v9'].astype(int) + merged['sn'].astype(int) + merged['base'].astype(int)
merged['Transported'] = votes >= 2

final = merged[['PassengerId', 'Transported']]
final.to_csv('submission_v10_majority.csv', index=False)

# Agreement check
total = len(merged)
agree_v9   = int((merged['v9']   == merged['Transported']).sum())
agree_sn   = int((merged['sn']   == merged['Transported']).sum())
agree_base = int((merged['base'] == merged['Transported']).sum())

print(f'Saved submission_v10_majority.csv  (Public-LB 0.81529)')
print(f'Total test rows           : {total}')
print(f'v9 agrees with majority   : {agree_v9}  ({agree_v9/total*100:.1f}%)')
print(f'sn agrees with majority   : {agree_sn}  ({agree_sn/total*100:.1f}%)')
print(f'base agrees with majority : {agree_base}  ({agree_base/total*100:.1f}%)')
final.head()

## 11. Summary

On this dataset and feature set, soft voting and stacking inflate OOF accuracy but fail to beat the simpler hard-voting ensemble. The OOF-to-LB gap is about 1% for weighted blend and about 0.4% for hard voting.

In [ ]:
summary_rows = [
 # Summary rows
    ('CatBoost (single)',                       oof_summary.loc[oof_summary.Model == 'CatBoost',             'OOF Acc @ Best Thr'].values[0],            np.nan),
    ('LightGBM (single)',                       oof_summary.loc[oof_summary.Model == 'LightGBM',             'OOF Acc @ Best Thr'].values[0],            np.nan),
    ('XGBoost (single)',                        oof_summary.loc[oof_summary.Model == 'XGBoost',              'OOF Acc @ Best Thr'].values[0],            np.nan),
    ('ExtraTrees (single)',                     oof_summary.loc[oof_summary.Model == 'ExtraTrees',           'OOF Acc @ Best Thr'].values[0],            np.nan),
    ('HistGradientBoosting (single)',           oof_summary.loc[oof_summary.Model == 'HistGradientBoosting', 'OOF Acc @ Best Thr'].values[0],            np.nan),
    ('① Weighted Blend (soft voting)',         blend_acc,  0.80874),
    ('② Stacking (LR meta-learner)',           stack_acc,  0.80500),
    ('③ Hard Voting / 3-way Majority ⭐',       np.nan,     0.81529),
]
summary_df = pd.DataFrame(summary_rows, columns=['Strategy', 'OOF Acc', 'Public LB'])
summary_df

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))
labels = summary_df['Strategy']
oof = summary_df['OOF Acc']
lb  = summary_df['Public LB']
xpos = np.arange(len(labels))

ax.bar(xpos - 0.18, oof.fillna(0), width=0.36, label='OOF Acc',  color='#5b9bd5')
ax.bar(xpos + 0.18, lb.fillna(0),  width=0.36, label='Public LB', color='#ed7d31')
ax.set_xticks(xpos)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylim(0.78, 0.83)
ax.set_ylabel('Accuracy')
ax.set_title('Spaceship Titanic — OOF vs Public LB across strategies')
ax.legend()
plt.tight_layout()
plt.savefig('oof_vs_lb_comparison.png', dpi=150)
plt.show()
print('Saved oof_vs_lb_comparison.png')

## Outputs

| File | Description | Public LB |
| --- | --- | --- |
| `submission_catboost.csv` | Single CatBoost (tuned) | - |
| `submission_lightgbm.csv` | Single LightGBM (tuned) | - |
| `submission_xgboost.csv` | Single XGBoost (tuned) | - |
| `submission_extratrees.csv` | Single ExtraTrees (tuned) | - |
| `submission_histgradientboosting.csv` | Single HistGradientBoosting (tuned) | - |
| `submission_weighted_blend.csv` | Soft Voting (Dirichlet-weighted) | 0.80874 |
| `submission_stacking.csv` | Stacked Generalization (LR meta) | 0.80500 |
| `submission_v10_majority.csv` | Hard Voting (3-way majority) | 0.81529 |

Recommended Kaggle submission: `submission_v10_majority.csv`.